# 🧠 NeuroScan-AI — CNN Training
## Binary Classification: Tumor / No Tumor
- Dataset: BraTS 2021 Task 1
- Model: CNN (TensorFlow/Keras)
- Tracking: MLflow + Dagshub

In [ ]:
!pip install dagshub mlflow nibabel -q

In [ ]:
!git clone https://github.com/kaushik-chariya/NeuroScan-AI.git
%cd NeuroScan-AI
!ls!pip install dagshub mlflow nibabel tensorflow==2.15.0 -q

In [ ]:
!git clone https://github.com/kaushik-chariya/NeuroScan-AI.git
%cd NeuroScan-AI

In [ ]:
%cd /kaggle/working/NeuroScan-AI
!ls

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
import mlflow

secrets = UserSecretsClient()
dagshub_token = secrets.get_secret("DAGSHUB_TOKEN")

os.environ["MLFLOW_TRACKING_USERNAME"] = "kaushik-chariya"
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

mlflow.set_tracking_uri("https://dagshub.com/kaushik-chariya/NeuroScan-AI.mlflow")
mlflow.set_experiment("CNN-Brain-Tumor-Classification")

print("✅ Connected!", mlflow.get_tracking_uri())

In [ ]:
import tarfile
import os

TAR_PATH = "/kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar"
EXTRACT_PATH = "/kaggle/working/BraTS2021"

os.makedirs(EXTRACT_PATH, exist_ok=True)

print("📦 Extracting dataset...")
with tarfile.open(TAR_PATH, "r") as tar:
    tar.extractall(EXTRACT_PATH)

print("✅ Done!")
print(f"Total patients: {len(os.listdir(EXTRACT_PATH))}")

In [ ]:
DATASET_PATH = "/kaggle/working/BraTS2021"

patients = os.listdir(DATASET_PATH)
print(f"✅ Total patients: {len(patients)}")
print(f"Sample: {patients[:3]}")

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split
import mlflow
import mlflow.keras

print(f"✅ TensorFlow: {tf.__version__}")
print(f"✅ GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
def load_patient_slice(patient_path: str, slice_indices: list = list(range(60, 130, 10))):
    """
    Load multiple 2D slices from T1ce, T2, FLAIR modalities.
    Returns list of (3-channel image, label) per slice.
    """
    files = os.listdir(patient_path)
    
    # Teen modalities
    def get_file(mod):
        f = [x for x in files if mod in x and x.endswith(".nii.gz")]
        return f[0] if f else None
    
    t1ce_f = get_file("t1ce")
    t2_f   = get_file("t2")
    flair_f = get_file("flair")
    seg_f  = get_file("seg")
    
    if not all([t1ce_f, t2_f, flair_f, seg_f]):
        return []
    
    t1ce_vol  = nib.load(os.path.join(patient_path, t1ce_f)).get_fdata()
    t2_vol    = nib.load(os.path.join(patient_path, t2_f)).get_fdata()
    flair_vol = nib.load(os.path.join(patient_path, flair_f)).get_fdata()
    seg_vol   = nib.load(os.path.join(patient_path, seg_f)).get_fdata()
    
    results = []
    for idx in slice_indices:
        if idx >= t1ce_vol.shape[2]:
            continue
        
        t1ce_s  = t1ce_vol[:, :, idx]
        t2_s    = t2_vol[:, :, idx]
        flair_s = flair_vol[:, :, idx]
        seg_s   = seg_vol[:, :, idx]
        
        # 3-channel image (T1ce, T2, FLAIR)
        img = np.stack([t1ce_s, t2_s, flair_s], axis=-1)
        label = 1 if np.any(seg_s > 0) else 0
        
        results.append((img, label))
    
    return results

# Test karo
sample_patient = os.path.join(DATASET_PATH, patients[0])
results = load_patient_slice(sample_patient)
img, label = results[0]
print(f"✅ Slice shape: {img.shape} | Label: {label}")
print(f"✅ Slices per patient: {len(results)}")

In [ ]:
plt.figure(figsize=(12, 4))

titles = ["T1ce", "T2", "FLAIR"]
for i in range(3):
    plt.subplot(1, 3, i+1)
    plt.imshow(img[:, :, i], cmap="gray")
    plt.title(f"{titles[i]} | {'Tumor' if label == 1 else 'No Tumor'}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2

IMAGE_SIZE = (128, 128)
X, y = [], []

print("🔄 Loading dataset...")

for i, patient in enumerate(patients):  # Pura 1252 patients
    patient_path = os.path.join(DATASET_PATH, patient)
    
    try:
        results = load_patient_slice(patient_path)  # Returns list of (img, label)
        
        if not results:
            continue
        
        for img_slice, label in results:
            
            # Normalize per channel
            for c in range(3):
                ch = img_slice[:, :, c]
                img_slice[:, :, c] = (ch - ch.min()) / (ch.max() - ch.min() + 1e-8)
            
            # Resize (3 channel already)
            img_resized = cv2.resize(img_slice, IMAGE_SIZE)
            
            X.append(img_resized)
            y.append(label)
        
    except Exception as e:
        print(f"⚠️  Skipping {patient}: {e}")
        continue
    
    if (i + 1) % 100 == 0:
        print(f"   Processed {i+1}/{len(patients)}")

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print(f"\n✅ Dataset ready!")
print(f"   X shape : {X.shape}")
print(f"   y shape : {y.shape}")
print(f"   Tumor   : {int(y.sum())} | No Tumor: {int(len(y) - y.sum())}")

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)
print(f"✅ Train : {X_train.shape[0]} samples")
print(f"✅ Val   : {X_val.shape[0]} samples")
print(f"✅ Test  : {X_test.shape[0]} samples")

In [ ]:
from tensorflow.keras import layers, models

def build_cnn_model(input_shape=(128, 128, 3), num_classes=1):
    inputs = layers.Input(shape=input_shape)
    
    # Block 1
    x = layers.Conv2D(32, (3,3), padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)
    
    # Block 2
    x = layers.Conv2D(64, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.25)(x)
    
    # Block 3
    x = layers.Conv2D(128, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.30)(x)
    
    # Block 4
    x = layers.Conv2D(256, (3,3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Dropout(0.40)(x)
    
    # Classifier
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.50)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(num_classes, activation='sigmoid')(x)
    
    model = models.Model(inputs, outputs)
    return model

model = build_cnn_model()
model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Class weights calculate karo
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print(f"✅ Class weights: {class_weight_dict}")

# Compile — Custom CNN ke liye
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
print("✅ Model compiled!")

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=8,
        restore_best_weights=True, mode='max'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', factor=0.3,
        patience=4, min_lr=1e-7, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/cnn_best.h5",
        monitor='val_auc', save_best_only=True,
        mode='max', verbose=1
    )
]
print("✅ Callbacks ready!")

In [ ]:
EPOCHS     = 30
BATCH_SIZE = 16

with mlflow.start_run(run_name="CNN-BraTS2021-V2"):
    # Log params
    mlflow.log_params({
        "epochs":             EPOCHS,
        "batch_size":         BATCH_SIZE,
        "learning_rate":      1e-4,
        "optimizer":          "Adam",
        "loss":               "binary_crossentropy",
        "image_size":         "128x128",
        "architecture":       "Custom CNN Deep",
        "modalities":         "T1ce+T2+FLAIR",
        "slices_per_patient": 7,
        "dataset":            "BraTS2021",
        "train_samples":      len(X_train),
        "val_samples":        len(X_val)
    })

    # Train
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        class_weight=class_weight_dict,
        verbose=1
    )

    # Log metrics per epoch
    for epoch in range(len(history.history["loss"])):
        mlflow.log_metrics({
            "train_loss":     history.history["loss"][epoch],
            "train_accuracy": history.history["accuracy"][epoch],
            "train_auc":      history.history["auc"][epoch],
            "val_loss":       history.history["val_loss"][epoch],
            "val_accuracy":   history.history["val_accuracy"][epoch],
            "val_auc":        history.history["val_auc"][epoch],
        }, step=epoch)

    # Final test evaluation
    test_loss, test_acc, test_auc = model.evaluate(X_test, y_test, verbose=0)
    mlflow.log_metrics({
        "test_loss":     test_loss,
        "test_accuracy": test_acc,
        "test_auc":      test_auc
    })

    print(f"\n✅ Test Accuracy : {test_acc:.4f}")
    print(f"✅ Test AUC      : {test_auc:.4f}")
    print(f"✅ Test Loss     : {test_loss:.4f}")

    # Save model
    model.save("/kaggle/working/cnn_model.h5")
    mlflow.log_artifact("/kaggle/working/cnn_model.h5")
    print("✅ Model saved + logged to MLflow!")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Train Acc')
axes[1].plot(history.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Accuracy')
axes[1].legend()

axes[2].plot(history.history['auc'], label='Train AUC')
axes[2].plot(history.history['val_auc'], label='Val AUC')
axes[2].set_title('AUC')
axes[2].legend()

plt.tight_layout()
plt.savefig("/kaggle/working/cnn_training_history.png")
plt.show()
print("✅ Plot saved!")

In [ ]:
# Test on one sample
sample = X_test[0:1]
pred = model.predict(sample, verbose=0)[0][0]
actual = int(y_test[0])

print(f"✅ Prediction  : {'Tumor' if pred > 0.5 else 'No Tumor'}")
print(f"✅ Confidence  : {pred*100:.2f}%")
print(f"✅ Actual Label: {'Tumor' if actual == 1 else 'No Tumor'}")

# 3 modalities dikhao
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
titles = ["T1ce", "T2", "FLAIR"]
for i in range(3):
    axes[i].imshow(sample[0][:, :, i], cmap='gray')
    axes[i].set_title(f"{titles[i]}\nPred: {'Tumor' if pred > 0.5 else 'No Tumor'} ({pred*100:.1f}%) | Actual: {'Tumor' if actual else 'No Tumor'}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import FileLink

print("📥 Download your model:")
display(FileLink('/kaggle/working/cnn_model.h5'))
display(FileLink('/kaggle/working/cnn_training_history.png'))
print("✅ Training Complete! Check Dagshub for MLflow metrics.")

In [ ]:
import os
print("✅ Model exists:", os.path.exists('/kaggle/working/cnn_model.h5'))
print("✅ Size:", os.path.getsize('/kaggle/working/cnn_model.h5'), "bytes")